# DA6401 — Continue Training from Friend's Checkpoints

**BEFORE RUNNING:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Click **Save Version** → **Save & Run All (Commit)**

Downloads weights from the shared Drive folder, runs 10 more epochs at low LR.

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -3
%cd /kaggle/working/repo
!pip install -q wandb albumentations gdown scikit-learn

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available()

In [ ]:
import gdown, os, shutil

# ── Download entire friend's Drive folder ──
FOLDER_ID = "1i7sCecWqc7rB68_JsPftJ6c22ex33JjL"

print("Downloading all files from Drive folder...")
downloaded = gdown.download_folder(id=FOLDER_ID, output="/kaggle/working/friends_weights", quiet=False)

# List what we got
print("\nFiles downloaded:")
folder_files = os.listdir("/kaggle/working/friends_weights")
for f in sorted(folder_files):
    mb = os.path.getsize(f"/kaggle/working/friends_weights/{f}") / 1e6
    print(f"  {f:40s}  {mb:.0f} MB")

In [ ]:
import os, shutil

# Auto-map files to classifier / localizer / unet by name or size
# Sizes: classifier ~516MB, localizer ~465MB, unet ~170MB
# Adjust the mapping below if file names are different

folder = "/kaggle/working/friends_weights"
files = {f: os.path.getsize(f"{folder}/{f}") / 1e6 for f in os.listdir(folder)}

# Try to match by name first, then by size
def find_file(keywords, size_range):
    """Find a file by keyword in name, or by size range (MB)."""
    for name, mb in sorted(files.items(), key=lambda x: -x[1]):
        if any(k.lower() in name.lower() for k in keywords):
            return name, mb
    for name, mb in sorted(files.items(), key=lambda x: -x[1]):
        if size_range[0] <= mb <= size_range[1]:
            return name, mb
    return None, 0

cls_name,  cls_mb  = find_file(["classif", "cls"],       (400, 600))
loc_name,  loc_mb  = find_file(["local",   "loc"],       (300, 500))
unet_name, unet_mb = find_file(["unet",    "seg", "net"], (50,  300))

print(f"Mapped:")
print(f"  classifier  ← {cls_name}  ({cls_mb:.0f} MB)")
print(f"  localizer   ← {loc_name}  ({loc_mb:.0f} MB)")
print(f"  unet        ← {unet_name}  ({unet_mb:.0f} MB)")

# Copy to checkpoints with standard names
for src_name, dst_name in [(cls_name,  "classifier.pth"),
                            (loc_name,  "localizer.pth"),
                            (unet_name, "unet.pth")]:
    if src_name:
        shutil.copy(f"{folder}/{src_name}", f"{CKPT}/{dst_name}")
        print(f"  ✓ Copied → {dst_name}")
    else:
        print(f"  ✗ Could not find file for {dst_name} — check names above")

In [ ]:
from data.pets_dataset import download_oxford_pet, create_dataloaders
download_oxford_pet("./data/oxford_pet")
train_loader, val_loader, _ = create_dataloaders(
    root="./data/oxford_pet", batch_size=32, num_workers=2
)
print("Dataset ready.")

## 1. Check baselines on friend's weights

In [ ]:
import torch
from sklearn.metrics import f1_score
from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from models.segmentation import VGG11UNet
from train import compute_iou, compute_dice_score

device = torch.device("cuda")

print("="*55)
print("  FRIEND'S WEIGHTS — BASELINE SCORES")
print("="*55)

# Classification
try:
    cls_model = VGG11Classifier(num_classes=37, dropout_p=0.5, use_bn=True).to(device)
    cls_model.load_state_dict(torch.load(f"{CKPT}/classifier.pth", map_location=device, weights_only=False))
    cls_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for b in val_loader:
            preds.extend(cls_model(b["image"].to(device)).argmax(1).cpu().tolist())
            labels.extend(b["label"].tolist())
    f1 = f1_score(labels, preds, average="macro")
    print(f"  Classification  val F1  : {f1:.4f}  (need ≥ 0.80)")
except Exception as e:
    print(f"  Classification  LOAD ERROR: {e}")

# Localization
try:
    loc_model = VGG11Localizer().to(device)
    loc_model.load_state_dict(torch.load(f"{CKPT}/localizer.pth", map_location=device, weights_only=False))
    loc_model.eval()
    iou_sum, iou50, total = 0.0, 0, 0
    with torch.no_grad():
        for b in val_loader:
            ious = compute_iou(loc_model(b["image"].to(device)).cpu(), b["bbox"])
            iou_sum += ious.sum().item(); iou50 += (ious >= 0.5).sum().item(); total += len(ious)
    print(f"  Localization    val IoU : {iou_sum/total:.4f},  Acc@0.5 = {iou50/total*100:.1f}%  (need ≥ 60%)")
except Exception as e:
    print(f"  Localization    LOAD ERROR: {e}")

# Segmentation
try:
    seg_model = VGG11UNet(num_classes=3).to(device)
    seg_model.load_state_dict(torch.load(f"{CKPT}/unet.pth", map_location=device, weights_only=False))
    seg_model.eval()
    dice_sum, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            pred = seg_model(b["image"].to(device)).argmax(1).cpu()
            dice_sum += compute_dice_score(pred, b["mask"]) * b["image"].size(0); total += b["image"].size(0)
    print(f"  Segmentation    val Dice: {dice_sum/total:.4f}  (need ≥ 0.30)")
except Exception as e:
    print(f"  Segmentation    LOAD ERROR: {e}")

print("="*55)

## 2. Continue Localizer (+10 epochs at lr=1e-4)

In [ ]:
import torch, time
import torch.nn as nn
from models.localization import VGG11Localizer
from losses.iou_loss import IoULoss
from train import compute_iou
import wandb

EXTRA_EPOCHS = 10
LR = 1e-4
device = torch.device("cuda")

model = VGG11Localizer().to(device)
model.load_state_dict(torch.load(f"{CKPT}/localizer.pth", map_location=device, weights_only=False))

# Baseline
model.eval()
iou_sum, total = 0.0, 0
with torch.no_grad():
    for b in val_loader:
        ious = compute_iou(model(b["image"].to(device)).cpu(), b["bbox"])
        iou_sum += ious.sum().item(); total += len(ious)
baseline_iou = iou_sum / total
print(f"Baseline val IoU: {baseline_iou:.4f}")

wandb.init(project="da6401-assignment2", name="localizer_friends_continued",
           tags=["continue", "localization"],
           config={"extra_epochs": EXTRA_EPOCHS, "lr": LR})

mse_loss  = nn.MSELoss()
iou_loss  = IoULoss(reduction="mean")
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EXTRA_EPOCHS, eta_min=1e-6)

best_iou = baseline_iou
for epoch in range(1, EXTRA_EPOCHS + 1):
    model.train()
    t0 = time.time()
    for b in train_loader:
        imgs, bboxes = b["image"].to(device), b["bbox"].to(device)
        optimizer.zero_grad()
        pred = model(imgs)
        (mse_loss(pred, bboxes) + iou_loss(pred, bboxes)).backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    iou_sum, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            ious = compute_iou(model(b["image"].to(device)).cpu(), b["bbox"])
            iou_sum += ious.sum().item(); total += len(ious)
    val_iou = iou_sum / total
    wandb.log({"val/iou": val_iou})
    print(f"  Epoch {epoch}/{EXTRA_EPOCHS} | Val IoU: {val_iou:.4f} | {time.time()-t0:.0f}s")
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), f"{CKPT}/localizer.pth")
        print(f"    ✓ Saved!")

wandb.finish()
print(f"\nLocalizer: {baseline_iou:.4f} → {best_iou:.4f}")

## 3. Continue U-Net (+10 epochs at lr=5e-4, partial freeze)

In [ ]:
from models.segmentation import VGG11UNet
from train import compute_dice_score

EXTRA_EPOCHS = 10
LR = 5e-4

seg_model = VGG11UNet(num_classes=3).to(device)
seg_model.load_state_dict(torch.load(f"{CKPT}/unet.pth", map_location=device, weights_only=False))

# Partial freeze: blocks 0-2 frozen
for name, param in seg_model.encoder.named_parameters():
    param.requires_grad = not any(name.startswith(b) for b in
                                  ["block0","block1","block2","pool0","pool1","pool2"])
print(f"Trainable: {sum(p.numel() for p in seg_model.parameters() if p.requires_grad):,}")

# Baseline
seg_model.eval()
dice_sum, total = 0.0, 0
with torch.no_grad():
    for b in val_loader:
        pred = seg_model(b["image"].to(device)).argmax(1).cpu()
        dice_sum += compute_dice_score(pred, b["mask"]) * b["image"].size(0); total += b["image"].size(0)
baseline_dice = dice_sum / total
print(f"Baseline val Dice: {baseline_dice:.4f}")

wandb.init(project="da6401-assignment2", name="unet_friends_continued",
           tags=["continue", "segmentation"],
           config={"extra_epochs": EXTRA_EPOCHS, "lr": LR})

criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0,1.0,2.0]).to(device))
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, seg_model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EXTRA_EPOCHS, eta_min=1e-6)

best_dice = baseline_dice
for epoch in range(1, EXTRA_EPOCHS + 1):
    seg_model.train()
    t0 = time.time()
    for b in train_loader:
        imgs, masks = b["image"].to(device), b["mask"].to(device)
        optimizer.zero_grad()
        criterion(seg_model(imgs), masks).backward()
        optimizer.step()
    scheduler.step()

    seg_model.eval()
    dice_sum, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            pred = seg_model(b["image"].to(device)).argmax(1).cpu()
            dice_sum += compute_dice_score(pred, b["mask"]) * b["image"].size(0); total += b["image"].size(0)
    val_dice = dice_sum / total
    wandb.log({"val/dice": val_dice})
    print(f"  Epoch {epoch}/{EXTRA_EPOCHS} | Val Dice: {val_dice:.4f} | {time.time()-t0:.0f}s")
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(seg_model.state_dict(), f"{CKPT}/unet.pth")
        print(f"    ✓ Saved!")

wandb.finish()
print(f"\nU-Net: {baseline_dice:.4f} → {best_dice:.4f}")

## 4. Summary — what to do next

In [ ]:
print("="*55)
print("  OUTPUT FILES")
print("="*55)
for f in sorted(os.listdir(CKPT)):
    mb = os.path.getsize(f"{CKPT}/{f}") / 1e6
    print(f"  {f:40s}  {mb:.0f} MB")
print()
print("  NEXT STEPS:")
print("  1. Download localizer.pth + unet.pth from the Output tab")
print("  2. Upload to Google Drive → share → copy file IDs")
print("  3. Paste the IDs here → update models/multitask.py")
print("  4. Rebuild ZIP → resubmit to Gradescope")
print("="*55)